**1. Import Libraries and Mount to Drive**

In [1]:
# import libraries
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from huggingface_hub import login
from datasets import load_dataset, get_dataset_config_names, concatenate_datasets
from transformers import AutoTokenizer
from google.colab import drive

# mount to personal google drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Download & Export Dataset**

In [2]:
# print message to show dataset processing has started for medcalc
print(f"\n--- Processing dataset: MedCalc-Bench ---")

# load only the train split
train_medcalc = load_dataset("ncbi/MedCalc-Bench-v1.0", split="train")

# formats each dataset row into instruction, input, output format
def format_medcalc(sample):

    # consistent task instruction for all samples
    instruction = (
        "Solve the following clinical calculation. Extract the necessary variables "
        "from the patient note, show the step-by-step reasoning and formula used, "
        "and provide the final numeric result."
    )

    # combines patient note and question into one input
    input_text = (
        f"Patient Note: {sample['Patient Note']}\n\n"
        f"Question: {sample['Question']}"
    )

    # uses the ground truth explanation as the reasoning output and appends the final answer
    output_text = (
        f"{sample['Ground Truth Explanation']}\n\n"
        f"Final Answer: ### {sample['Ground Truth Answer']} ###"
    )

    return {
        "instruction":    instruction,
        "input":          input_text,
        "output":         output_text,
        "dataset_source": "MedCalc_Bench"
    }

# applies the formatting to the dataset
train_medcalc = train_medcalc.map(format_medcalc)

# ensure dataset contains the correct column order
column_order = ["instruction", "input", "output", "dataset_source"]
train_medcalc = train_medcalc.select_columns(column_order)

print(f"--- Total Number of Rows: {len(train_medcalc)} ---")
print(f"--- Final Columns: {train_medcalc.column_names} ---")

# converts the dataset to jsonl format and saves file to storage
output_file = "medcalc_training_data.jsonl"
train_medcalc.to_json(output_file, lines=True, force_ascii=False)

# confirm the file has been saved and show its name
print(f"--- Completed! Saved to {output_file}")


--- Processing dataset: MedCalc-Bench ---


README.md:   0%|          | 0.00/2.11k [00:00<?, ?B/s]

train_data.csv:   0%|          | 0.00/40.9M [00:00<?, ?B/s]

test_data.csv:   0%|          | 0.00/4.00M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10053 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1047 [00:00<?, ? examples/s]

Map:   0%|          | 0/10053 [00:00<?, ? examples/s]

--- Total Number of Rows: 10053 ---
--- Final Columns: ['instruction', 'input', 'output', 'dataset_source'] ---


Creating json from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

--- Completed! Saved to medcalc_training_data.jsonl


**3. Remove Duplicates**

In [3]:
# loads the json file into a variable called ds
ds = load_dataset("json", data_files="medcalc_training_data.jsonl", split="train")

# turns the dataset into a panda table
df = pd.DataFrame(ds)

# creates another copy of the dataframe
unprocessed_duplicates_df = df.copy()

# gets the length of the unprocessed dataframe
unprocessed_duplicates_ds = len(unprocessed_duplicates_df)

# finds duplicate rows and gets their index (row number) based on content columns only (ignoring source column)
duplicated_rows_unprocessed = unprocessed_duplicates_df.index[unprocessed_duplicates_df.duplicated(subset=["instruction", "input", "output"])].tolist()

# gets the length of the duplicate rows
duplicates_unprocessed = len(duplicated_rows_unprocessed)

# drops the duplicated rows based on content columns only (ignoring source column)
processed_duplicates_df = unprocessed_duplicates_df.drop_duplicates(subset=["instruction", "input", "output"])

# gets the length of the processed dataframe
processed_duplicates_ds = len(processed_duplicates_df)

# prints a summary of the deduplication process
print(f"---  Deduplication Summary ---")
print(f"    - Rows before : {unprocessed_duplicates_ds}")
print(f"    - Duplicates found : {duplicates_unprocessed}")
print(f"    - Rows after : {processed_duplicates_ds}")
print(f"    - Rows removed : {unprocessed_duplicates_ds - processed_duplicates_ds}")

Generating train split: 0 examples [00:00, ? examples/s]

---  Deduplication Summary ---
    - Rows before : 10053
    - Duplicates found : 95
    - Rows after : 9958
    - Rows removed : 95


**4. Remove Missing Rows (instruction, input & output)**

In [4]:
# creates another copy of the dataframe
unprocessed_missing_df = processed_duplicates_df.copy()

# gets the length of the unprocessed dataframe
unprocessed_missing_ds = len(unprocessed_missing_df)

# finds empty or missing rows in instruction, input, and output columns
empty_rows_unprocessed = unprocessed_missing_df.index[
    unprocessed_missing_df['instruction'].isnull() | unprocessed_missing_df['instruction'].astype(str).str.strip().eq("") |
    unprocessed_missing_df['input'].isnull() | unprocessed_missing_df['input'].astype(str).str.strip().eq("") |
    unprocessed_missing_df['output'].isnull() | unprocessed_missing_df['output'].astype(str).str.strip().eq("")
].tolist()

# count empty rows per column within the unprocessed dataset
empty_instruction_rows_unprocessed = (unprocessed_missing_df['instruction'].isnull() | unprocessed_missing_df['instruction'].astype(str).str.strip().eq("")).sum()
empty_input_rows_unprocessed = (unprocessed_missing_df['input'].isnull() | unprocessed_missing_df['input'].astype(str).str.strip().eq("")).sum()
empty_output_rows_unprocessed = (unprocessed_missing_df['output'].isnull() | unprocessed_missing_df['output'].astype(str).str.strip().eq("")).sum()

# gets the length of the missing rows
missing_row_count_unprocessed = len(empty_rows_unprocessed)

# drops the empty or missing rows from instruction, input, and output columns
processed_missing_df = unprocessed_missing_df.drop(index=empty_rows_unprocessed).reset_index(drop=True)

# gets the length of the processed dataframe
processed_missing_ds = len(processed_missing_df)

# count empty rows per column in the processed dataset
empty_instruction_rows_processed = (processed_missing_df['instruction'].isnull() | processed_missing_df['instruction'].astype(str).str.strip().eq("")).sum()
empty_input_rows_processed = (processed_missing_df['input'].isnull() | processed_missing_df['input'].astype(str).str.strip().eq("")).sum()
empty_output_rows_processed = (processed_missing_df['output'].isnull() | processed_missing_df['output'].astype(str).str.strip().eq("")).sum()

# prints a summary of the missing rows removal process
print(f"--- Missing Rows Summary ---")
print(f"    - Rows before : {unprocessed_missing_ds}")
print(f"    - Missing instruction (before) : {empty_instruction_rows_unprocessed}")
print(f"    - Missing input (before) : {empty_input_rows_unprocessed}")
print(f"    - Missing output (before) : {empty_output_rows_unprocessed}")
print(f"    - Rows removed : {missing_row_count_unprocessed}")
print(f"    - Missing instruction (after) : {empty_instruction_rows_processed}")
print(f"    - Missing input (after) : {empty_input_rows_processed}")
print(f"    - Missing output (after) : {empty_output_rows_processed}")
print(f"    - Rows after : {processed_missing_ds}")

--- Missing Rows Summary ---
    - Rows before : 9958
    - Missing instruction (before) : 0
    - Missing input (before) : 0
    - Missing output (before) : 0
    - Rows removed : 0
    - Missing instruction (after) : 0
    - Missing input (after) : 0
    - Missing output (after) : 0
    - Rows after : 9958


**5. Remove AI Mentions**

In [5]:
# creates a copy of the dataframe
unprocessed_ai_text_df = processed_missing_df.copy()

# list of ai patterns to remove
ai_patterns = [
    r'\bas an ai\b',
    r'\bas a language model\b'
]

# combines all columns into one string per row
combined = unprocessed_ai_text_df[['instruction', 'input', 'output']].astype(str).agg(' '.join, axis=1)

# count per pattern before removal
pattern_counts = {pattern: combined.str.contains(pattern, case=False, na=False).sum() for pattern in ai_patterns}

# drop rows that match any pattern and reset index
mask = combined.str.contains('|'.join(ai_patterns), case=False, na=False)
processed_ai_text_df = unprocessed_ai_text_df[~mask].reset_index(drop=True)

# get counts for specific phrases using regex
as_an_ai_count = pattern_counts[r'\bas an ai\b']
language_model_count = pattern_counts[r'\bas a language model\b']

# prints a summary of the ai words detection process
print(f"--- AI Words Detection Summary ---")
print(f"    - Dataset Rows (before)                       : {len(unprocessed_ai_text_df)}")
print(f"    - as an ai detected (before)                  : {as_an_ai_count}")
print(f"    - as a language model detected (before)       : {language_model_count}")
print(f"    - Rows removed in total                       : {len(unprocessed_ai_text_df) - len(processed_ai_text_df)}")
print(f"    - Dataset Rows (after)                        : {len(processed_ai_text_df)}")

--- AI Words Detection Summary ---
    - Dataset Rows (before)                       : 9958
    - as an ai detected (before)                  : 0
    - as a language model detected (before)       : 0
    - Rows removed in total                       : 0
    - Dataset Rows (after)                        : 9958


**6. Remove Rows, Over Token Limit**

In [6]:
# load the Llama-3.2-3B-Instruct tokenizer to check token lengths
llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

# max allowed tokens
max_tokens = 2048

# creates a copy of the dataframe
unprocessed_llama_tokens_df = processed_ai_text_df.copy()

# gets the length of the unprocessed dataframe
unprocessed_llama_tokens_ds = len(unprocessed_llama_tokens_df)

# empty system prompt
system_prompt = ""

# function that get the total token count for a single row
def get_token_length(sample):
    # get instruction text
    user_prompt = str(sample["instruction"])
    # get input text if it exists
    input_text = sample.get("input", "")

    # add input to instruction if it's not empty
    if pd.notna(input_text) and str(input_text).strip():
        user_prompt = user_prompt + "\n\n" + str(input_text).strip()

    # structure the data as a list of messages
    messages = [
        {"role": "user", "content": system_prompt + user_prompt if system_prompt else user_prompt},
        {"role": "assistant", "content": str(sample['output'])}
    ]

    # use the auto template
    chat_string = llama_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokens = llama_tokenizer.encode(chat_string)

    return len(tokens)

# calculate token lengths for all rows
token_lengths = unprocessed_llama_tokens_df.apply(get_token_length, axis=1)

# keep only rows with equal to or less then 2048
processed_llama_tokens_df = unprocessed_llama_tokens_df[token_lengths <= max_tokens].reset_index(drop=True)

# gets the length of the processed dataframe
processed_llama_tokens_ds = len(processed_llama_tokens_df)

# calculate how many rows were dropped
dropped_rows = unprocessed_llama_tokens_ds - processed_llama_tokens_ds

# prints a summary of the token filter process
print(f"--- Llama-3.2-3B-Instruct Token Filter Summary ---")
print(f"    - Dataset Rows (before)                : {unprocessed_llama_tokens_ds}")
print(f"    - Longest row found (tokens)           : {token_lengths.max()}")
print(f"    - Rows above 2048 tokens removed        : {dropped_rows}")
print(f"    - Dataset Rows (after)                 : {processed_llama_tokens_ds}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

--- Llama-3.2-3B-Instruct Token Filter Summary ---
    - Dataset Rows (before)                : 9958
    - Longest row found (tokens)           : 3200
    - Rows above 2048 tokens removed        : 340
    - Dataset Rows (after)                 : 9618


In [7]:
# load the Qwen2.5-3B-Instruct tokenizer to check token lengths
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

# max allowed tokens
max_tokens = 2048

# creates a copy of the dataframe
unprocessed_qwen_tokens_df = processed_llama_tokens_df.copy()

# gets the length of the unprocessed dataframe
unprocessed_qwen_tokens_ds = len(unprocessed_qwen_tokens_df)

# empty system prompt
system_prompt = ""

# function that get the total token count for a single row
def get_qwen_token_length(sample):
    # get instruction text
    user_prompt = str(sample["instruction"])
    # get input text if it exists
    input_text = sample.get("input", "")

    # add input to instruction if it's not empty
    if pd.notna(input_text) and str(input_text).strip():
        user_prompt = user_prompt + "\n\n" + str(input_text).strip()

    # structure the data as a list of messages
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": str(sample['output'])}
    ]

    # use the auto template
    chat_string = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokens = qwen_tokenizer.encode(chat_string)

    return len(tokens)

# calculate token lengths for all rows
qwen_token_lengths = unprocessed_qwen_tokens_df.apply(get_qwen_token_length, axis=1)

# keep only rows with equal to or less then 2048
processed_qwen_tokens_df = unprocessed_qwen_tokens_df[qwen_token_lengths <= max_tokens].reset_index(drop=True)

# gets the length of the processed dataframe
processed_qwen_tokens_ds = len(processed_qwen_tokens_df)

# calculate how many rows were dropped
dropped_rows_qwen = unprocessed_qwen_tokens_ds - processed_qwen_tokens_ds

# prints a summary of the token filter process
print(f"--- Qwen2.5-3B-Instruct Token Filter Summary ---")
print(f"    - Dataset Rows (before)                : {unprocessed_qwen_tokens_ds}")
print(f"    - Longest row found (tokens)           : {qwen_token_lengths.max()}")
print(f"    - Rows above 2048 tokens removed        : {dropped_rows_qwen}")
print(f"    - Dataset Rows (after)                 : {processed_qwen_tokens_ds}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

--- Qwen2.5-3B-Instruct Token Filter Summary ---
    - Dataset Rows (before)                : 9618
    - Longest row found (tokens)           : 2176
    - Rows above 2048 tokens removed        : 87
    - Dataset Rows (after)                 : 9531


**7. Download Random Sample**

In [8]:
# prints 10 random rows from the final dataset to verify data quality
samples = processed_qwen_tokens_df.sample(10, random_state=42)

# opens a text file to save the samples
with open('medical_math_sample_verification.txt', 'w') as f:
    for i, row in samples.iterrows():
        f.write(f"\n--- Sample {i+1} ---\n")
        f.write(f"    - Instruction : {row['instruction']}\n")
        f.write(f"    - Input : {row['input']}\n")
        f.write(f"    - Output : {row['output']}\n")
        f.write(f"    - Subset : {row['dataset_source']}\n")
        f.write(f"\n{'='*80}\n")

# prints confirmation of the save
print("--- Saved to medical_math_sample_verification.txt ---")

--- Saved to medical_math_sample_verification.txt ---


**8. Import Medical Dataset & Calculate Math Data Percentages**

In [9]:
# folder path to the medical dataset files
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/Med-MathInstruct/Medical-Data-Only"

# load the train, val, and test sets for the medical dataset
med_train = pd.read_json(f"{folder_path}/mv_train.jsonl", lines=True)
med_val   = pd.read_json(f"{folder_path}/mv_val.jsonl", lines=True)
med_test  = pd.read_json(f"{folder_path}/mv_test.jsonl", lines=True)

# calculate the total size of the medical dataset
medical_dataset = len(med_train) + len(med_val) + len(med_test)

# print the medical dataset sizes and splits
print(f"--- Medical Dataset Summary ---")
print(f"    - Final Medical Dataset Size : {medical_dataset}")
print(f"    - Medical Train Set Size : {len(med_train)}")
print(f"    - Medical Validation Set Size : {len(med_val)}")
print(f"    - Medical Test Set Size : {len(med_test)}")

print("\n")

# print the amount of mathematical data needed per split (15% of each)
print(f"--- Math Data Needed ---")
print(f"    - Mathematical data needed to be added to train: {int(len(med_train) * 0.15)}")
print(f"    - Mathematical data needed to be added to val: {int(len(med_val) * 0.15)}")
print(f"    - Mathematical data needed to be added to test: {int(len(med_test) * 0.15)}")

--- Medical Dataset Summary ---
    - Final Medical Dataset Size : 59300
    - Medical Train Set Size : 53370
    - Medical Validation Set Size : 2965
    - Medical Test Set Size : 2965


--- Math Data Needed ---
    - Mathematical data needed to be added to train: 8005
    - Mathematical data needed to be added to val: 444
    - Mathematical data needed to be added to test: 444


**9. Split Mathematical Dataset to 15% Percentage**

In [10]:
# 15% of each medical split size
train_size = 8005
val_size = 444
test_size = 444

# split the dataset into train, val, and test using the allocated sizes
math_train, temp = train_test_split(processed_qwen_tokens_df, train_size=train_size, random_state=42)
math_val, math_test = train_test_split(temp, train_size=val_size, test_size=test_size, random_state=42)

# print the math dataset split sizes
print(f"--- Math Dataset Summary ---")
print(f"    - Math Train Set Size : {len(math_train)}")
print(f"    - Math Val Set Size : {len(math_val)}")
print(f"    - Math Test Set Size : {len(math_test)}")

--- Math Dataset Summary ---
    - Math Train Set Size : 8005
    - Math Val Set Size : 444
    - Math Test Set Size : 444


**10. Combine Medical and Mathematical Datasets Together**

In [11]:
# combine the medical and math sets together
combined_train = pd.concat([med_train, math_train], ignore_index=True)
combined_val = pd.concat([med_val, math_val], ignore_index=True)
combined_test = pd.concat([med_test, math_test], ignore_index=True)

# shuffle each set so med and math are mixed (random_state=42 for consistency)
combined_train = combined_train.sample(frac=1, random_state=42).reset_index(drop=True)
combined_val = combined_val.sample(frac=1, random_state=42).reset_index(drop=True)
combined_test = combined_test.sample(frac=1, random_state=42).reset_index(drop=True)

# print a summary of the final combined splits
print(f"--- Final Dataset Split Summary ---")
print(f"    - Combined Train: {len(combined_train)}")
print(f"    - Combined Val:   {len(combined_val)}")
print(f"    - Combined Test:  {len(combined_test)}")

--- Final Dataset Split Summary ---
    - Combined Train: 61375
    - Combined Val:   3409
    - Combined Test:  3409


**11. Calculate Prestnages of Subsets**

In [12]:
# combine all splits into one full dataset
full_combined_df = pd.concat([combined_train, combined_val, combined_test], ignore_index=True)

# counts the number of rows for each dataset source
alpacare_ds = (full_combined_df['dataset_source'] == 'AlpaCare-52k').sum()
biomed_ds = (full_combined_df['dataset_source'] == 'Bioinstruct').sum()
hendrycks_ds = (full_combined_df['dataset_source'] == 'MedCalc_Bench').sum()

# gets the total number of rows in the final dataset
final_ds = len(full_combined_df)

# calculates the percentage of each dataset source
alpacare_percentage = (alpacare_ds / final_ds) * 100
biomed_percentage = (biomed_ds / final_ds) * 100
hendrycks_percentage = (hendrycks_ds / final_ds) * 100

# prints a summary of the dataset subset percentages
print(f"--- Dataset Subset Percentages ---")
print(f"    - AlpaCare-52k Size : {alpacare_ds}")
print(f"    - Bioinstruct Size : {biomed_ds}")
print(f"    - Hendrycks Math Size : {hendrycks_ds}")
print(f"    - AlpaCare-52k Percentage : {round(alpacare_percentage, 2)}%")
print(f"    - Bioinstruct Percentage : {round(biomed_percentage, 2)}%")
print(f"    - MedCalc_Bench Percentage : {round(hendrycks_percentage, 2)}%")
print(f"    - Final Size : {final_ds}")

--- Dataset Subset Percentages ---
    - AlpaCare-52k Size : 34295
    - Bioinstruct Size : 25005
    - Hendrycks Math Size : 8893
    - AlpaCare-52k Percentage : 50.29%
    - Bioinstruct Percentage : 36.67%
    - MedCalc_Bench Percentage : 13.04%
    - Final Size : 68193


**12. Remove Column (subset_sorce) for Mathematical Dataset**

In [13]:
# Remove the column called 'dataset_source' from train set
combined_train = combined_train.drop('dataset_source', axis=1, errors='ignore')
combined_val = combined_val.drop('dataset_source', axis=1, errors='ignore')
combined_test = combined_test.drop('dataset_source', axis=1, errors='ignore')

# prints confirmation
print(f"--- Dataset Remove Column subset_source ---")
print(f"    - Train Columns: {combined_train.columns.tolist()}")
print(f"    - Val Columns:   {combined_val.columns.tolist()}")
print(f"    - Test Columns:  {combined_test.columns.tolist()}")

--- Dataset Remove Column subset_source ---
    - Train Columns: ['instruction', 'input', 'output']
    - Val Columns:   ['instruction', 'input', 'output']
    - Test Columns:  ['instruction', 'input', 'output']


**13. Export the Final Dataset (Medical + Mathematical Version)**

In [14]:
# folder path to save to
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/Med-MathInstruct"
# creates folder if does not exist
os.makedirs(folder_path, exist_ok=True)

# deletes existing files before saving
for f in ["combined_medical_math_train.jsonl", "combined_medical_math_val.jsonl", "combined_medical_math_test.jsonl"]:
    file_path = f"{folder_path}/{f}"
    if os.path.exists(file_path):
        os.remove(file_path)

# exports the data to the file location on google drive
combined_train.to_json(f"{folder_path}/combined_medical_math_train.jsonl", orient="records", lines=True)
combined_val.to_json(f"{folder_path}/combined_medical_math_val.jsonl", orient="records", lines=True)
combined_test.to_json(f"{folder_path}/combined_medical_math_test.jsonl", orient="records", lines=True)

# print confirmation that the files have been saved
print(f"--- Mathematical Version Files Saved ---")
print(f"    - Saved mathematical version training dataset successfully")
print(f"    - Saved mathematical version val dataset successfully")
print(f"    - Saved mathematical version test dataset successfully")

--- Mathematical Version Files Saved ---
    - Saved mathematical version training dataset successfully
    - Saved mathematical version val dataset successfully
    - Saved mathematical version test dataset successfully
